# Kaggle: Download Audio from Google Drive (Public Folder)

**Audio folder:** https://drive.google.com/drive/folders/1-KbBhuAVJYrOrOp5Ab39EoSf82piSIDr

**⚠️ IMPORTANT: Make folder public FIRST:**
1. Open the link above
2. Right-click the `audio` folder inside
3. Share > Anyone with link > Viewer > Copy link
4. Replace FOLDER_ID below with the new ID

**Then run all cells.**

In [ ]:
# === STEP 1: INSTALL DEPENDENCIES ===
!pip install -q gdown librosa numpy pandas scikit-learn tqdm

import os
import subprocess

OUTPUT_DIR = '/kaggle/working/audio'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Dependencies installed!')

In [ ]:
# === STEP 2: DOWNLOAD FROM GOOGLE DRIVE ===
# REPLACE THIS with your public folder ID
# After making the folder public, copy the ID from the URL
# URL looks like: https://drive.google.com/drive/folders/XXXXX?usp=sharing
# XXXXX is the folder ID

FOLDER_ID = '1-KbBhuAVJYrOrOp5Ab39EoSf82piSIDr'  # REPLACE THIS

print(f'Downloading from folder: {FOLDER_ID}')

# Method 1: gdown (works with public folders)
!gdown --folder https://drive.google.com/drive/folders/$FOLDER_ID -O /kaggle/working/audio --recursive --remainingQuota 100

# Count files
audio_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.m4a')]
print(f'\nDownloaded {len(audio_files)} audio files')

In [ ]:
# === STEP 3: EXTRACT FEATURES ===
import numpy as np
import librosa
from tqdm import tqdm

def extract_features(audio_path, sr=22050):
    try:
        y, _ = librosa.load(audio_path, sr=sr, mono=True)
        
        hop_length = 512
        rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
        times = librosa.times_like(rms, sr=sr, hop_length=hop_length)
        silence_mask = rms < 0.01
        
        utterances = []
        in_utt = False
        
        for i, (t, is_sil) in enumerate(zip(times, silence_mask)):
            if not is_sil and not in_utt:
                start_idx = i
                in_utt = True
            elif is_sil and in_utt:
                end_idx = i
                start_t = times[start_idx]
                end_t = times[end_idx]
                
                if end_t - start_t > 0.3:
                    rms_vals = rms[start_idx:end_idx]
                    
                    start_sample = int(start_t * sr)
                    end_sample = int(end_t * sr)
                    segment = y[start_sample:end_sample]
                    
                    if len(segment) < 1024:
                        in_utt = False
                        continue
                    
                    feat = []
                    
                    # Energy
                    feat.extend([
                        np.mean(rms_vals),
                        np.max(rms_vals),
                        np.std(rms_vals),
                        np.max(rms_vals) / (np.mean(rms_vals) + 1e-8)
                    ])
                    
                    # ZCR
                    zcr = librosa.feature.zero_crossing_rate(segment)[0]
                    feat.extend([np.mean(zcr), np.std(zcr)])
                    
                    # MFCCs
                    mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
                    for i in range(13):
                        feat.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
                    
                    # Spectral
                    spec_cent = librosa.feature.spectral_centroid(y=segment, sr=sr)[0]
                    spec_bw = librosa.feature.spectral_bandwidth(y=segment, sr=sr)[0]
                    spec_rolloff = librosa.feature.spectral_rolloff(y=segment, sr=sr)[0]
                    feat.extend([np.mean(spec_cent), np.std(spec_cent)])
                    feat.extend([np.mean(spec_bw), np.std(spec_bw)])
                    feat.extend([np.mean(spec_rolloff), np.std(spec_rolloff)])
                    
                    label = 1 if np.max(rms_vals) > 0.15 else 0
                    
                    utterances.append({
                        'features': np.array(feat, dtype=np.float32),
                        'label': label
                    })
                
                in_utt = False
        
        return utterances
    except:
        return []

print('Extracting features from all audio files...')
all_samples = []

for audio_file in tqdm(audio_files):
    audio_path = f'{OUTPUT_DIR}/{audio_file}'
    samples = extract_features(audio_path)
    all_samples.extend(samples)

print(f'\nProcessed {len(audio_files)} files, {len(all_samples)} utterances')

In [ ]:
# === STEP 4: TRAIN MODEL ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
import pickle

X = np.array([s['features'] for s in all_samples])
y = np.array([s['label'] for s in all_samples])

print(f'Dataset: {len(X)} samples, pos={y.sum()} ({100*y.mean():.1f}%)')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')

# Save
with open('/kaggle/working/energy_model.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'scaler': scaler, 'f1': f1}, f)
print('\nModel saved to /kaggle/working/energy_model.pkl')